# 03 — Context Managers and the `with` Statement

Any time a resource needs **guaranteed cleanup** — files, locks, network connections, temp directories, timing measurements, env vars — a context manager is the tool.

The headline guarantee: the cleanup runs **even if an exception is raised** inside the block. That's `try/finally` with less boilerplate.

## The shape: `with ... as ...:`

The classic example — a file that auto-closes:

In [1]:
import tempfile, os

# Write something to a temp file so we have data to read.
tmp = tempfile.NamedTemporaryFile("w", delete=False)
tmp.write("hello\nworld\n")
tmp.close()

# Idiomatic read — file is closed automatically on the way out.
with open(tmp.name) as f:
    text = f.read()
print(text)
print("file closed?", f.closed)

os.unlink(tmp.name)

hello
world

file closed? True


## What `with` desugars to

```python
with cm as x:
    body
```

is roughly:

```python
x = cm.__enter__()
try:
    body
finally:
    cm.__exit__(exc_type, exc_value, traceback)
```

(Strictly: `__exit__` can return `True` to suppress the exception. That's rare.)

## Building one — the `__enter__` / `__exit__` form

Pick this form when you need real state (multiple methods, instance attrs).

In [2]:
import time

class Timed:
    def __init__(self, label: str):
        self.label = label
        self.elapsed = 0.0

    def __enter__(self):
        self._start = time.perf_counter()
        # Whatever __enter__ returns becomes the `as` value.
        return self

    def __exit__(self, exc_type, exc, tb):
        self.elapsed = time.perf_counter() - self._start
        # Print regardless of success/failure — this is `finally` behavior.
        outcome = "raised" if exc_type else "ok"
        print(f"[{self.label}] {self.elapsed * 1000:.2f} ms ({outcome})")
        # Return None / False to let the exception propagate.
        # Returning True would SWALLOW it — rarely what you want.

with Timed("sum-1M") as t:
    total = sum(range(1_000_000))
print("elapsed after exit:", t.elapsed)

# Still runs on exception:
try:
    with Timed("fail-block"):
        raise RuntimeError("boom")
except RuntimeError as e:
    print("caught outside:", e)

[sum-1M] 54.61 ms (ok)
elapsed after exit: 0.05460999999195337
[fail-block] 0.01 ms (raised)
caught outside: boom


## The shorter form — `@contextmanager`

When you don't need a class — just "do setup, yield, do teardown" — `contextlib.contextmanager` lets you write it as a generator function. Cleaner for one-shot resources.

In [3]:
import time
from contextlib import contextmanager

@contextmanager
def timed(label: str):
    # Everything BEFORE `yield` is __enter__.
    start = time.perf_counter()
    try:
        yield                          # body of `with` runs here
    finally:
        # Everything AFTER `yield` (inside finally) is __exit__.
        elapsed = time.perf_counter() - start
        print(f"[{label}] {elapsed * 1000:.2f} ms")

with timed("loop"):
    sum(range(1_000_000))

# Still runs on exception (because we used try/finally inside):
try:
    with timed("explosion"):
        raise RuntimeError("boom")
except RuntimeError:
    print("caught outside")

[loop] 48.62 ms
[explosion] 0.01 ms
caught outside


**Rule for `@contextmanager`:** put the teardown in a `finally` block. Without it, an exception inside the `with` skips your cleanup.

### Yielding a value to bind with `as`

In [4]:
import os
from contextlib import contextmanager

@contextmanager
def temp_env(**vars):
    """Set env vars on enter, restore them on exit (even if the block raises)."""
    saved = {k: os.environ.get(k) for k in vars}
    os.environ.update({k: str(v) for k, v in vars.items()})
    try:
        yield dict(vars)              # value bound by `as`
    finally:
        # Restore: re-set what we saved; delete keys that weren't there before.
        for k, original in saved.items():
            if original is None:
                os.environ.pop(k, None)
            else:
                os.environ[k] = original

with temp_env(MY_FLAG="1", MY_LEVEL="debug") as set_vars:
    print("inside :", os.environ.get("MY_FLAG"), os.environ.get("MY_LEVEL"))
    print("set vars:", set_vars)

print("after  :", os.environ.get("MY_FLAG"), os.environ.get("MY_LEVEL"))

inside : 1 debug
set vars: {'MY_FLAG': '1', 'MY_LEVEL': 'debug'}
after  : None None


## Useful helpers from `contextlib`

In [5]:
from contextlib import suppress, redirect_stdout
import io, os

# 1. `suppress` — like an `except` you don't have to write.
with suppress(FileNotFoundError):
    os.remove("nope_does_not_exist.txt")
print("continued past the missing file")

# 2. `redirect_stdout` — capture print output into a buffer.
buf = io.StringIO()
with redirect_stdout(buf):
    print("this goes into buf, not the screen")
print("captured:", buf.getvalue().rstrip())

continued past the missing file
captured: this goes into buf, not the screen


## Multiple context managers in one `with`

In [6]:
import tempfile, os

# Create two temp files so we have something to open.
p1 = tempfile.NamedTemporaryFile("w", delete=False); p1.write("hello\n"); p1.close()
p2 = tempfile.NamedTemporaryFile("w", delete=False); p2.write("world\n"); p2.close()

# Open both at once; both auto-close.
with open(p1.name) as f1, open(p2.name) as f2:
    print(f1.read().strip(), f2.read().strip())

os.unlink(p1.name); os.unlink(p2.name)

hello world


### `ExitStack` — when the count is dynamic

If you don't know how many resources you'll open until runtime, use `contextlib.ExitStack`.

In [7]:
import tempfile, os
from contextlib import ExitStack

paths = []
for i in range(3):
    t = tempfile.NamedTemporaryFile("w", delete=False)
    t.write(f"file {i}\n")
    t.close()
    paths.append(t.name)

with ExitStack() as stack:
    files = [stack.enter_context(open(p)) for p in paths]
    for f in files:
        print(f.read().strip())
    # All files closed automatically when the with block exits.

for p in paths:
    os.unlink(p)

file 0


file 1
file 2


## Where this shows up later in the curriculum

- **`with httpx.Client() as client:`** — HTTP clients in the agentic AI folders.
- **`async with`** — same shape for async resources (folder 11).
- **FastAPI `lifespan`** (folder 14) is a context manager: setup → yield → teardown.
- **pytest fixtures** with `yield` (folder 09) follow exactly this pattern.

## Mini-exercise

1. Convert `Timed` (class-based) above into `@contextmanager`-style `timed_v2(label)`. Make it `yield` an object with an `elapsed` attribute readable *outside* the block.
2. Write `chdir(path)` as a context manager: switch to `path` on enter, restore the original CWD on exit (use `os.getcwd()` / `os.chdir()`). Verify it restores even when the block raises.
3. Build `acquired(lock)` that calls `lock.acquire()` on enter and `lock.release()` on exit. Use `threading.Lock`. Why is the `__exit__`/`finally` form essential here?
4. Predict: does this leak the file handle? Why?
   ```python
   f = open("data.txt")
   try:
       do_thing(f)
   except Exception:
       pass
   # forgot f.close()
   ```
   Rewrite it with `with`.

In [8]:
# your solutions here


**Takeaways**

- `with cm as x:` runs `cm.__enter__()` → block → `cm.__exit__(...)`. Cleanup runs even on exceptions.
- Use the **class form** when you need real state; use **`@contextmanager`** for setup/yield/teardown.
- Always put the teardown in a `finally` inside the generator — otherwise exceptions skip it.
- `contextlib` has ready-made helpers: `suppress`, `redirect_stdout`, `ExitStack`.

Next: [04_file_pipeline/](04_file_pipeline/) — exceptions + context managers in a real .py module.